<h2>Import Library</h2>

In [4]:
import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

<h2>Load Data & Pelabelan</h2>

In [6]:
df = pd.read_csv('dataset_scraping_netflix.csv')
df

,content,score
0,woiii terikat janji eps 20 kenapa kok blank sa...,1
1,jelek,1
2,"kenapa saya tidak bisa menonton,beberapa hari ...",1
3,langganan bukannya murah. diatas 50k. tapi ga ...,1
4,Netflix ke Napa dah film gak bisa di putar,1
...,...,...
3594,"film nya ga lengkap, mahal",1
3595,"@🐢aaz🕸🕸, a, aazzazaz",1
3596,aplikasi apa coba ga bisa di daftar,1
3597,"iiu, ada uii",4


In [8]:
slang_dict = {
    "yg": "yang", "gak": "tidak", "ga": "tidak", "gk": "tidak", "tdk": "tidak",
    "bgt": "banget", "mantap": "bagus", "mantuuul": "bagus", "oke": "bagus",
    "ok": "bagus", "aplikasinya": "aplikasi", "lemot": "lambat", 
    "kecewa": "buruk", "sip": "bagus", "jlk": "jelek", "gembel": "buruk"
}

def normalize_slang(text):
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    return " ".join(normalized_words)

In [10]:
df = df.dropna()
df

,content,score,label
0,woiii terikat janji eps 20 kenapa kok blank sa...,1,0
1,jelek,1,0
2,"kenapa saya tidak bisa menonton,beberapa hari ...",1,0
3,langganan bukannya murah. diatas 50k. tapi ga ...,1,0
4,Netflix ke Napa dah film gak bisa di putar,1,0
...,...,...,...
3594,"film nya ga lengkap, mahal",1,0
3595,"@🐢aaz🕸🕸, a, aazzazaz",1,0
3596,aplikasi apa coba ga bisa di daftar,1,0
3597,"iiu, ada uii",4,1


In [11]:
df = df[df['score'] != 3]
df['label'] = df['score'].apply(lambda x: 1 if x > 3 else 0)

<h2>Preprocessing Data</h2>

In [13]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|@\S+|#\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = normalize_slang(text)
    text = stemmer.stem(text)
    return text

df['ulasan'] = df['content'].apply(clean_text)
df = df[df['ulasan'].str.strip() != '']

<h2>Ekstraksi Fitur</h2>

In [14]:
X = df['ulasan']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

<h2>Pelatihan & Evaluasi Model</h2>

In [19]:
model = SVC(kernel='linear', C=1.0)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
akurasi = accuracy_score(y_test, y_pred)


print(f"Akurasi Testing : {akurasi * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Negatif', 'Positif']))

Akurasi Testing : 88.39%
              precision    recall  f1-score   support

     Negatif       0.89      0.96      0.92       494
     Positif       0.86      0.67      0.75       178

    accuracy                           0.88       672
   macro avg       0.87      0.82      0.84       672
weighted avg       0.88      0.88      0.88       672

